**Task**  
Assume we have labeled image.  
Assume we have a prediction image based on our model.  
Assume these images are binary--if there is a shrub then 1, otherwise 0. The goal here is to simply compare shrubs, not their genus/other attributes. That is a separate task and for that model to be accurate it must inherently be good at this task already.  
  
Using the labeled image we can produce label masks for the shrubs in that image. 
    
**Plan**  


using evaluate_resolutions_requirements.txt dependencies

In [93]:
import numpy as np
class ShrubObject:
    def __init__(self, mask):
        self.mask = mask

    def get_mask(self):
        return self.mask

    def find_center(self, shrub_value: int|np.ndarray = 1):
        """
        Finds the center of shrub along each direction.

        Args:
            shrub_value: int for grayscale images
                         np array for more than one color

        Ex. (2d binary)
        pred_mask_2d = np.array([
            [0,1,0,0],
            [1,1,1,1],
            [0,1,0,0]
        ])
        
        s2 = ShrubObject(np.expand_dims(pred_mask_2d, axis=-1)) # need to expand last dim for grayscale. last dim is assumed to be a color scale.
        s2.find_center()

        Output: array([1.        , 1.33333333])

        Ex. (2d rgb)
        pred_mask_2d_rgb = np.array([
            [[0,0,0],[255,255,255],[0,0,0],[0,0,0]],
            [[255,255,255],[255,255,255],[255,255,255],[255,255,255]],
            [[0,0,0],[255,255,255],[0,0,0],[0,0,0]]
        ])
        
        shrub_value = np.array([255,255,255])
        s2_rgb = ShrubObject(pred_mask_2d_rgb)
        s2_rgb.find_center(shrub_value)

        Output: array([1.        , 1.33333333])

        Ex. (3d binary)
        pred_mask_3d = np.array([
            [[0,0,0,0],
            [0,1,0,0],
            [0,0,0,0]],
            [[0,1,0,0],
            [1,1,1,1],
            [0,1,0,0]],
            [[0,0,0,0],
            [0,1,0,0],
            [0,0,0,0]]
        ])

        s3 = ShrubObject(np.expand_dims(pred_mask_3d, axis=-1))
        s3.find_center()

        Output: array([1.  , 1.  , 1.25])

        """
        indices = np.argwhere(np.all(self.mask == shrub_value, axis=-1))
        center = indices.T.mean(axis=-1)

        return center
    
    def get_count_type(self, shrub_value: int|np.ndarray = 1):
        """
        Counts the number of shrub-related pixels in the mask.

        Args:
            shrub_value - shrub pixel/voxel color tile. 
                          grayscale must be its own color dimension.
        """
        Xel_count = (self.mask == shrub_value).sum()
        other_count = (self.mask != shrub_value).sum()
        return Xel_count, other_count

    def get_count_difference(label_shrub, pred_shrub: ShrubObject, shrub_value: int|np.ndarray): 
        """
        Finds shrub and non-shrub count differences. Assumes classical sets are provided, not fuzzy sets, for the pixel/voxel matching.

        Args:
            label_shrub (self) - current shrub object to take difference relative to
            pred_shrub - ShrubObject to take difference against
            shrub_value - int or np array representing shrub color scale value.
        
        Returns:
            n_diff - a list of the form [difference in shrub counts, difference in non-shrub counts]

        Ex.
        pred_mask_2d = np.array([
            [0,1,0,0],
            [1,1,1,1],
            [0,1,0,0]
        ])

        inv_pred_mask_2d = 1 - pred_mask_2d

        s2 = ShrubObject(np.expand_dims(pred_mask_2d, axis=-1))
        label_s2 = ShrubObject(np.expand_dims(inv_pred_mask_2d, axis=-1))
        label_s2.get_count_difference(s2, 1)

        Output: [np.int64(0), np.int64(0)]
        """
        label_counts = label_shrub.get_count_type(shrub_value)
        pred_counts = pred_shrub.get_count_type(shrub_value)
        n_diff = [lc - pc for lc, pc in zip(label_counts, pred_counts)]
        return n_diff
    
    def get_count_correctness(label_shrub, pred_shrub: ShrubObject, shrub_value: int|np.ndarray):
        """
        Returns the number of correct and incorrect predictions for shrub and non-shrub classifications.

        Args:
            label_shrub (self) - current shrub object to treat as label
            pred_shrub - ShrubObject to compare against
            shrub_value - int or np array representing shrub color scale value.

        Returns:
            n_tp - number of true positives
            n_fn - number of false negatives
            n_tn - number of true negatives
            n_fp - number of false positives

        Ex.
        pred_mask_2d = np.array([
            [0,1,1,0],
            [1,1,1,1],
            [0,1,0,0]
        ])

        inv_pred_mask_2d = 1 - pred_mask_2d

        s2 = ShrubObject(np.expand_dims(pred_mask_2d, axis=-1))
        label_s2 = ShrubObject(np.expand_dims(inv_pred_mask_2d, axis=-1))
        label_s2.get_count_correctness(s2, 1)

        Output: (np.int64(0), np.int64(5), np.int64(0), np.int64(7))
        """
        # actually shrub
        n_tp = ((label_shrub.get_mask() == shrub_value) & (pred_shrub.get_mask() == shrub_value)).sum()
        n_fn = ((label_shrub.get_mask() == shrub_value) & (pred_shrub.get_mask() != shrub_value)).sum()
        # actually not shrub
        n_tn = ((label_shrub.get_mask() != shrub_value) & (pred_shrub.get_mask() != shrub_value)).sum()
        n_fp = ((label_shrub.get_mask() != shrub_value) & (pred_shrub.get_mask() == shrub_value)).sum()

        return n_tp, n_fn, n_tn, n_fp

In [94]:
pred_mask_2d = np.array([
    [0,1,1,0],
    [1,1,1,1],
    [0,1,0,0]
])

inv_pred_mask_2d = 1 - pred_mask_2d

s2 = ShrubObject(np.expand_dims(pred_mask_2d, axis=-1))
# s2.find_center()
# s2.get_count_type(1)
label_s2 = ShrubObject(np.expand_dims(inv_pred_mask_2d, axis=-1))
label_s2.get_count_difference(s2, 1)
label_s2.get_count_correctness(s2, 1)

(np.int64(0), np.int64(5), np.int64(0), np.int64(7))